data load(world happiness) and import library

In [1]:
import kagglehub
import shutil
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Download latest version
path = kagglehub.dataset_download("unsdsn/world-happiness")

# data 저장 경로 설정 
target_path = "./Week_1"

# 기존에 같은 폴더가 있으면 삭제 후 덮어쓰기, 없으면 바로 복사
if os.path.exists(target_path):
    shutil.rmtree(target_path)
shutil.copytree(path, target_path)

print("새로운 데이터 저장 경로:", target_path)

새로운 데이터 저장 경로: ./Week_1


#### Context
155개의 국가에서 5월 20일 행복에 날에 측정한 happiness level에 대한 데이터다. 정부나 기관에서 정책을 만들기 위해서 측정을 계속 한다.
경제학, 심리학, 설문조사 분석, 국가 통계, 보건, 공공 정책 등 다양한 분야의 저명한 전문가들이 웰빙 측정 지표를 효과적으로 활용하여 국가의 발전 정도를 평가하는 방법을 설명합니다.

#### Content
Happiness Score: Gallup 세계 여론조사 데이터를 사용한다. 현재 삶에대한 만족도를 0~10으로 평가한다.
6가지 요소로 구성한다. economic production, social support, life expectancy, freedom, absence of corruption, and generosity
이러한 요소들은 각 국가의 총점에 영향을 미치지는 않지만, 일부 국가의 순위가 다른 국가보다 높은 이유를 설명해 준다.

#### Residual
설명되지 않은 구성 요소는 국가마다 다르며, 이는 6개 변수가 2014-2016년 평균 수명 평가를 과대 또는 과소 설명하는 정도를 반영한다. 
이러한 residual은 전체 국가 집합에서 평균적으로 약 0의 값을 가진다. 이 residual을 estimate life evaluation과 결합하여
항상 양수값을 가지게 한다.
일부 estimate life evaluation residual은 상당히 커서 0에서 10까지의 척도에서 1점을 넘는 경우도 있지만, 디스토피아에서 계산된 값(평균 생명이 0에서 10까지의 척도에서 1.85로 평가됨)보다는 항상 훨씬 작습니다.

#### Distopia
다른 국가의 행복 평가 하한선으로 쓰인다. Utopia와는 반대되는 개념이다.
Dystopia Residual metric은 실제로 디스토피아 행복 점수(1.85)에 이전 답변에서 언급한 각 국가의 residual 값 또는 unexplained value를 더한 값이다.

target이 명확한 Quantitive data이고 이에 영향을 미치는 6가지 predictor가 존재하므로 Multiple Linear regression이 합당해보인다.

먼저 데이터의 구조를 확인하기 위해 데이터 설명에 표시된 Distopia Residual이 명확하게 표시된 2015데이터를 먼저 확인한다.

In [3]:
# Week_1 폴더 안의 2015.csv 파일 경로 설정
csv_path = os.path.join("./Week_1", "2015.csv")

# 데이터 프레임으로 불러오기
df = pd.read_csv(csv_path)

# 데이터의 상위 5개 행을 확인하여 컬럼명과 값의 형태를 확인한다.
display(df.head())

# 데이터의 행과 열 개수 확인
print(f"데이터 크기 (행, 열): {df.shape}")

print("--- 데이터 정보 확인 ---")
df.info()

print("\n--- 결측치 개수 확인 ---")
print(df.isnull().sum())

print("\n--- 중복된 데이터 개수 확인 ---")
duplicates = df.duplicated().sum()
print(f"중복된 데이터 개수: {duplicates}")

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176


데이터 크기 (행, 열): (158, 12)
--- 데이터 정보 확인 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        158 non-null    object 
 1   Region                         158 non-null    object 
 2   Happiness Rank                 158 non-null    int64  
 3   Happiness Score                158 non-null    float64
 4   Standard Error                 158 non-null    float64
 5   Economy (GDP per Capita)       158 non-null    float64
 6   Family                         158 non-null    float64
 7   Health (Life Expectancy)       158 non-null    float64
 8   Freedom                        158 non-null    float64
 9   Trust (Government Corruption)  158 non-null    float64
 10  Generosity                     158 non-null    float64
 11  Dystopia Residual              158 non-null    float64
dtypes: floa

결축치가 없는 것을 확인했다. 다음으로는 기초통계량을 확인한다.

In [4]:
display(df.describe())

,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
count,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000,158.000000
mean,79.493671,5.375734,0.047885,0.846137,0.991046,0.630259,0.428615,0.143422,0.237296,2.098977
std,45.754363,1.145010,0.017146,0.403121,0.272369,0.247078,0.150693,0.120034,0.126685,0.553550
min,1.000000,2.839000,0.018480,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.328580
25%,40.250000,4.526000,0.037268,0.545808,0.856823,0.439185,0.328330,0.061675,0.150553,1.759410
50%,79.500000,5.232500,0.043940,0.910245,1.029510,0.696705,0.435515,0.107220,0.216130,2.095415
75%,118.750000,6.243750,0.052300,1.158448,1.214405,0.811013,0.549092,0.180255,0.309883,2.462415
max,158.000000,7.587000,0.136930,1.690420,1.402230,1.025250,0.669730,0.551910,0.795880,3.602140


Linear regression을 하기 위해서 Country, Region은 문자열 데이터이고 필요 없기 때문에 제외한다. Happiness Rank또한 Happiness Score의 정답임으로 제외한다. 성능 평가를 위해 5-Fold CV를 사용해본다.

In [5]:
# 1. 사용할 Features(독립변수 X)와 Target(종속변수 y) 정의
features = [
    'Economy (GDP per Capita)', 
    'Family', 
    'Health (Life Expectancy)', 
    'Freedom', 
    'Trust (Government Corruption)', 
    'Generosity',
    'Dystopia Residual' 
]

X = df[features].values # CV에 넣기 위해 numpy array로 변환
y = df['Happiness Score'].values

# 2. 5-Fold CV 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 성능 기록을 위한 리스트
mse_scores = []
r2_scores = []

# 3. K-Fold 교차 검증 루프 (5번 반복)
fold_num = 1
for train_index, test_index in kf.split(X):
    # a. 데이터 분할
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # b. 스케일링 (각 Fold마다 독립적으로 수행!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # c. 모델 학습
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # d. 예측 및 평가
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mse_scores.append(mse)
    r2_scores.append(r2)
    
    print(f"[Fold {fold_num}] MSE: {mse:.4f}, R2: {r2:.4f}")
    fold_num += 1

# 4. 최종 결과 (5번의 평균)
print("\n=== K-Fold 교차 검증 최종 결과 ===")
print(f"평균 MSE: {np.mean(mse_scores):.4f}")
print(f"평균 R2 Score: {np.mean(r2_scores):.4f}")

[Fold 1] MSE: 0.0000, R2: 1.0000
[Fold 2] MSE: 0.0000, R2: 1.0000
[Fold 3] MSE: 0.0000, R2: 1.0000
[Fold 4] MSE: 0.0000, R2: 1.0000
[Fold 5] MSE: 0.0000, R2: 1.0000

=== K-Fold 교차 검증 최종 결과 ===
평균 MSE: 0.0000
평균 R2 Score: 1.0000


모든 $R^{2}$ 값이 1.0이 나왔다. 이는 Happiness Score가 단순 덧셈으로만 만들어져있기 때문이다. Prediction이 아닌 단순 복원에 불과하다.
따라서 인위적인 보정값인 Dystopia Residual을 제외하고 모델을 다시 만든다.

In [6]:
# 1. 사용할 Features(독립변수 X)와 Target(종속변수 y) 정의
features = [
    'Economy (GDP per Capita)', 
    'Family', 
    'Health (Life Expectancy)', 
    'Freedom', 
    'Trust (Government Corruption)', 
    'Generosity',
]

X = df[features].values # CV에 넣기 위해 numpy array로 변환
y = df['Happiness Score'].values

# 2. 5-Fold CV 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 성능 기록을 위한 리스트
mse_scores = []
r2_scores = []

# 3. K-Fold 교차 검증 루프 (5번 반복)
fold_num = 1
for train_index, test_index in kf.split(X):
    # a. 데이터 분할
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # b. 스케일링 (각 Fold마다 독립적으로 수행!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # c. 모델 학습
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # d. 예측 및 평가
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mse_scores.append(mse)
    r2_scores.append(r2)
    
    print(f"[Fold {fold_num}] MSE: {mse:.4f}, R2: {r2:.4f}")
    fold_num += 1

# 4. 최종 결과 (5번의 평균)
print("\n=== K-Fold 교차 검증 최종 결과 ===")
print(f"평균 MSE: {np.mean(mse_scores):.4f}")
print(f"평균 R2 Score: {np.mean(r2_scores):.4f}")

[Fold 1] MSE: 0.2419, R2: 0.8295
[Fold 2] MSE: 0.3087, R2: 0.7609
[Fold 3] MSE: 0.2942, R2: 0.7217
[Fold 4] MSE: 0.4488, R2: 0.6835
[Fold 5] MSE: 0.3645, R2: 0.7050

=== K-Fold 교차 검증 최종 결과 ===
평균 MSE: 0.3316
평균 R2 Score: 0.7401


이제 각 predictor들이 Happiness에 대해 얼만큼 관여하고 있는지 coef를 통해 알아본다.

In [7]:
# 1. 사용할 Features 정의 (Dystopia Residual 제외)
features = [
    'Economy (GDP per Capita)', 
    'Family', 
    'Health (Life Expectancy)', 
    'Freedom', 
    'Trust (Government Corruption)', 
    'Generosity'
]

X = df[features]
y = df['Happiness Score']

# 2. 스케일링 (Standardization)
# 기여도(Coefficient) 크기를 공정하게 비교하기 위해 변수들의 단위를 통일합니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# pandas DataFrame으로 다시 변환 (statsmodels에서 컬럼명을 예쁘게 보기 위함)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# 3. 절편(Intercept) 추가
# statsmodels는 y절편을 수식에 자동으로 포함하지 않으므로 직접 명시해 주어야 합니다.
X_scaled_df = sm.add_constant(X_scaled_df)

# 4. OLS (Ordinary Least Squares) 다중 선형 회귀 모델 적합
model_sm = sm.OLS(y, X_scaled_df).fit()

# 5. 요약 결과 출력
print(model_sm.summary())

                            OLS Regression Results                            
Dep. Variable:        Happiness Score   R-squared:                       0.777
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     87.81
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           1.04e-46
Time:                        13:11:59   Log-Likelihood:                -126.46
No. Observations:                 158   AIC:                             266.9
Df Residuals:                     151   BIC:                             288.3
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

p-value값을 확인해보니 

Family (0.3825): 행복에 가장 큰 영향을 미침
Economy (0.3458): 두 번째로 큰 영향
Health (0.2402)
Freedom (0.2003)

상위 4개의 feature들만 사용해도 모델의 설명력이 보장될 것 같다. 이를 확인하기 위해서 4개의 feature만 사용해서 모델을 다시 fit 해본다.

In [8]:
# 1. pvalue로 확인한대로 Feature를 4개로 줄여본다
features = [
    'Economy (GDP per Capita)', 
    'Family', 
    'Health (Life Expectancy)', 
    'Freedom', 
]

X = df[features].values # CV에 넣기 위해 numpy array로 변환
y = df['Happiness Score'].values

# 2. 5-Fold CV 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 성능 기록을 위한 리스트
mse_scores = []
r2_scores = []

# 3. K-Fold 교차 검증 루프 (5번 반복)
fold_num = 1
for train_index, test_index in kf.split(X):
    # a. 데이터 분할
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # b. 스케일링 (각 Fold마다 독립적으로 수행!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # c. 모델 학습
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # d. 예측 및 평가
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mse_scores.append(mse)
    r2_scores.append(r2)
    
    print(f"[Fold {fold_num}] MSE: {mse:.4f}, R2: {r2:.4f}")
    fold_num += 1

# 4. 최종 결과 (5번의 평균)
print("\n=== K-Fold 교차 검증 최종 결과 ===")
print(f"평균 MSE: {np.mean(mse_scores):.4f}")
print(f"평균 R2 Score: {np.mean(r2_scores):.4f}")

[Fold 1] MSE: 0.2320, R2: 0.8364
[Fold 2] MSE: 0.2976, R2: 0.7694
[Fold 3] MSE: 0.2867, R2: 0.7288
[Fold 4] MSE: 0.4696, R2: 0.6689
[Fold 5] MSE: 0.3540, R2: 0.7135

=== K-Fold 교차 검증 최종 결과 ===
평균 MSE: 0.3280
평균 R2 Score: 0.7434


설명력이 약간이지만 올라간걸 확인할 수 있다. MSE는 줄었고 R2 Score는 조금이지만 올라갔다. 따라서 predictor 4개인 모델로 다른 연도의 데이터를 fit 해보려고 한다.
그전에 다른 연도의 cloumn명이 살짝 다르기 때문에 이름을 확인하기 위해 column을 확인한다.

In [9]:
# 폴더 내 파일 목록 확인
files = [f for f in os.listdir(target_path) if f.endswith('.csv')]
print(f"존재하는 데이터 파일: {files}\n")

# 각 연도별로 컬럼명 확인하기
for file in sorted(files):
    year = file.split('.')[0]
    df_temp = pd.read_csv(os.path.join(target_path, file))
    print(f"--- {year}년 컬럼명 ---")
    print(df_temp.columns.tolist())
    print("\n")

존재하는 데이터 파일: ['2015.csv', '2016.csv', '2017.csv', '2018.csv', '2019.csv']

--- 2015년 컬럼명 ---
['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']


--- 2016년 컬럼명 ---
['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Lower Confidence Interval', 'Upper Confidence Interval', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']


--- 2017년 컬럼명 ---
['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high', 'Whisker.low', 'Economy..GDP.per.Capita.', 'Family', 'Health..Life.Expectancy.', 'Freedom', 'Generosity', 'Trust..Government.Corruption.', 'Dystopia.Residual']


--- 2018년 컬럼명 ---
['Overall rank', 'Country or region', 'Score', 'GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choice

각 연도에 맞게 column명을 통합하고 데이터를 합친다.

In [10]:
# 연도별 컬럼명 매핑 딕셔너리
column_mapping = {
    # 2017년
    'Happiness.Score': 'Happiness Score',
    'Economy..GDP.per.Capita.': 'Economy (GDP per Capita)',
    'Family': 'Family',
    'Health..Life.Expectancy.': 'Health (Life Expectancy)',
    'Freedom': 'Freedom',
    
    # 2018년, 2019년
    'Score': 'Happiness Score',
    'GDP per capita': 'Economy (GDP per Capita)',
    'Social support': 'Family',
    'Healthy life expectancy': 'Health (Life Expectancy)',
    'Freedom to make life choices': 'Freedom',
    'Country or region': 'Country' # 2018, 2019는 국가 컬럼명도 다름
}

# 합친 데이터를 담을 빈 리스트
all_data = []

files = [f for f in os.listdir(target_path) if f.endswith('.csv')]

for file in sorted(files):
    year = int(file.split('.')[0])
    df_temp = pd.read_csv(os.path.join(target_path, file))
    
    # 컬럼명 통일
    df_temp = df_temp.rename(columns=column_mapping)
    
    # 'Year' 컬럼 추가 (어느 연도 데이터인지 표시)
    df_temp['Year'] = year
    
    # 우리가 필요한 핵심 4개 변수 + 타겟 변수 + 국가명 + 연도만 추출
    required_cols = [
        'Country', 'Year', 'Happiness Score', 
        'Economy (GDP per Capita)', 'Family', 
        'Health (Life Expectancy)', 'Freedom'
    ]
    
    # 만약 해당 연도에 필요한 컬럼이 다 있다면 리스트에 추가
    if all(col in df_temp.columns for col in required_cols):
        all_data.append(df_temp[required_cols])
    else:
        print(f"Warning: {year}년 데이터에 필요한 컬럼이 누락되었습니다.")

# 리스트에 모인 모든 데이터프레임을 위아래로 결합
combined_df = pd.concat(all_data, ignore_index=True)

print("=== 데이터 통합 완료 ===")
print(f"통합된 데이터 크기: {combined_df.shape}")
display(combined_df.head())
display(combined_df.tail())

=== 데이터 통합 완료 ===
통합된 데이터 크기: (782, 7)


,Country,Year,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom
0,Switzerland,2015,7.587,1.39651,1.34951,0.94143,0.66557
1,Iceland,2015,7.561,1.30232,1.40223,0.94784,0.62877
2,Denmark,2015,7.527,1.32548,1.36058,0.87464,0.64938
3,Norway,2015,7.522,1.45900,1.33095,0.88521,0.66973
4,Canada,2015,7.427,1.32629,1.32261,0.90563,0.63297


,Country,Year,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom
777,Rwanda,2019,3.334,0.359,0.711,0.614,0.555
778,Tanzania,2019,3.231,0.476,0.885,0.499,0.417
779,Afghanistan,2019,3.203,0.350,0.517,0.361,0.000
780,Central African Republic,2019,3.083,0.026,0.000,0.105,0.225
781,South Sudan,2019,2.853,0.306,0.575,0.295,0.010


In [11]:
# 1. 사용할 4가지 핵심 Features와 Target 설정
features_final = [
    'Economy (GDP per Capita)', 
    'Family', 
    'Health (Life Expectancy)', 
    'Freedom'
]

# 결측치가 혹시 합치는 과정에서 생겼을 수 있으므로 제거 (보수적인 접근)
combined_df_clean = combined_df.dropna(subset=features_final + ['Happiness Score'])

X_final = combined_df_clean[features_final]
y_final = combined_df_clean['Happiness Score']

# 2. 스케일링 (Standardization)
# 전체 5년 치 데이터를 기준으로 스케일링합니다.
scaler_final = StandardScaler()
X_final_scaled = scaler_final.fit_transform(X_final)

# DataFrame으로 변환 (statsmodels 사용을 위해)
X_final_scaled_df = pd.DataFrame(X_final_scaled, columns=features_final)

# 3. statsmodels를 위한 절편(Intercept) 추가
X_final_scaled_df = sm.add_constant(X_final_scaled_df)

# 4. 최종 모델 학습 (OLS)
final_model = sm.OLS(y_final.values, X_final_scaled_df).fit()

# 5. 최종 결과 요약 출력
print("=== 최종 모델 (2015-2019 통합 데이터) 결과 ===")
print(final_model.summary())

=== 최종 모델 (2015-2019 통합 데이터) 결과 ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.754
Model:                            OLS   Adj. R-squared:                  0.753
Method:                 Least Squares   F-statistic:                     594.8
Date:                Tue, 23 Jun 2026   Prob (F-statistic):          9.15e-235
Time:                        13:11:59   Log-Likelihood:                -654.84
No. Observations:                 782   AIC:                             1320.
Df Residuals:                     777   BIC:                             1343.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------

마지막으로 2024 대한민국의 데이터로 모델의 정확도를 확인해본다.

In [12]:
# 1. 2024년 대한민국 데이터 준비 (우리가 찾은 4개 핵심 변수)
korea_2024_data = {
    'Economy (GDP per Capita)': [1.815],
    'Family': [1.178],
    'Health (Life Expectancy)': [0.770],
    'Freedom': [0.555]
}

df_korea_2024 = pd.DataFrame(korea_2024_data)

# 2. 데이터 스케일링
# 주의: 새 데이터로 fit을 하면 절대 안 됩니다! 
# 기존 2015-2019 데이터로 만든 스케일러(scaler_final)의 기준(transform)을 그대로 사용해야 합니다.
korea_2024_scaled = scaler_final.transform(df_korea_2024)

# DataFrame으로 변환 및 컬럼명 지정
df_korea_2024_scaled = pd.DataFrame(korea_2024_scaled, columns=features_final)

# 3. statsmodels 예측을 위한 절편(const) 추가
df_korea_2024_scaled = sm.add_constant(df_korea_2024_scaled, has_constant='add')

# 4. 모델 예측 (Prediction)
predicted_score = final_model.predict(df_korea_2024_scaled)

# 5. 결과 비교
actual_score = 6.058  # 2024년 실제 행복 점수

print("=== 🇰🇷 2024년 대한민국 행복도 예측 결과 ===")
print(f"실제 2024년 행복 점수: {actual_score:.4f}")
print(f"모델이 예측한 행복 점수: {predicted_score.values[0]:.4f}")

# 오차 계산 (절대값)
error = abs(actual_score - predicted_score.values[0])
print(f"예측 오차(Error): {error:.4f}")

=== 🇰🇷 2024년 대한민국 행복도 예측 결과 ===
실제 2024년 행복 점수: 6.0580
모델이 예측한 행복 점수: 6.9460
예측 오차(Error): 0.8880


#### Overall
782개라는 엄청난 양의 데이터(5년 치 누적)를 넣었음에도 $R^2$가 0.754 (75.4%)가 나왔다.
보통 현실 세계의 복잡한 현상(국가별 행복도 같은 심리적/사회적 지표)을 다룰 때 4개의 변수만으로 75%의 설명력을 가진다는 것은 "이 4개 지표가 행복을 결정하는 핵심 지표"임을 강력하게 시사한다.

p-value가 모두 0.000이다. 이는 predictor들이 의미 있임을 확인한다.

#### 바뀐 포인트
* 2015 데이터만 사용했을 때는 1. Family, 2. Economy, 3. Health, 4. Freedom 순이었다.

* 하지만 통합된 데이터에서는 1. Economy, 2. Freedom, 3. Health, 4.Family 순이 되었다.

#### 결론
* 2015 한해만 봤을 때는 Family 수치가 행복의 가장 큰 조건처럼 보였지만, 5년 데이터를 합쳐보니 결국 GDP, Freedom이 중요했다.
반면에 Family의 영향력은 줄어들었다.

* 따라서 이 모델은 단순히 "돈과 내 삶을 스스로 선택할 수 있는 자유가 국가 행복도의 중심이다."는 메세지를 던지고 있다고 봐도 무방하다.

#### 한계
* 데이터를 어떻게 측정했는지가 가장 중요하다. GDP의 경우 경제의 객관적인 지표로 사용할 수 있지만, 자유도의 경우는 사람마다 주관적이다. 따라서 측정 방법에 따라 데이터가 변할 수 있다.

* 또한 2024 한국의 데이터를 보면 경제/건강 지표가 높음에도 실제 행복도는 낮게 나타나는 등 이 4개 변수(75%) 외에 설명되지 않는 25%의 영역(예: 과도한 경쟁, 스트레스, 근로 시간 등 국가별 특수한 사회문화적 요인)이 모델에 반영되지 못했다는 한계가 존재한다.
